# Building Agentic RAG with Agno + Google Gemini Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/17oH1AOCH23Nvb1A6mKrMbuFLezd71jui?usp=sharing)

This notebook demonstrates how to build a **Retrieval-Augmented Generation (RAG)** pipeline using:

- [**Agno**](https://github.com/agno-agi/agno) — A lightweight, high-performance framework for building multi-modal AI agents
- [**Google Gemini Embeddings**](https://ai.google.dev/gemini-api/docs/embeddings) — Google's latest embedding models (`gemini-embedding-001` and `gemini-embedding-2-preview`)
- [**Gemini 2.5 Flash**](https://ai.google.dev/gemini-api/docs) — Google's fast and capable generative model
- [**LanceDB**](https://lancedb.com/) — A serverless vector database (no infrastructure needed)

## What You'll Learn

1. Set up Agno agents with Google Gemini as the LLM
2. Use the latest Gemini Embedding models (`gemini-embedding-001` and `gemini-embedding-2-preview`) for vector search
3. Build a knowledge base from text and PDF documents
4. Create an **Agentic RAG** pipeline where the agent autonomously decides when to search
5. Explore **multimodal embeddings** with `gemini-embedding-2-preview` (text, images, and more)
6. Build a multi-agent team for complex research tasks

---

## 1. Installation & Setup

In [ ]:
!pip install -q "agno[google,lancedb]" lancedb google-genai ddgs pypdf pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 9.4 MB/s eta 0:00:00


In [ ]:
import os
from getpass import getpass

# Set your Google API key
# Get one free at: https://aistudio.google.com/apikey
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API Key: ")

print("Google API Key is set.")

Enter your Google API Key: ··········
Google API Key is set.


---
## 2. Quick Start — Your First Agno Agent with Gemini

Let's start with a simple agent powered by **Gemini 2.0 Flash**.

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini

agent = Agent(
    model=Gemini(id="gemini-2.5-flash"),
    description="You are a helpful AI research assistant.",
    instructions=["Be concise and accurate.", "Use markdown formatting."],
    markdown=True,
)

agent.print_response(
    "Explain Retrieval-Augmented Generation (RAG) in 3 bullet points.",
    stream=True,
)

---
## 3. Exploring Google Gemini Embeddings

Google's latest embedding models:

| Model | Dimensions | Max Input | Modalities | Status |
|---|---|---|---|---|
| `gemini-embedding-001` | 128–3,072 (default 768) | 2,048 tokens | Text only | **Stable** |
| `gemini-embedding-2-preview` | 128–3,072 (default 768) | 8,192 tokens | **Text, Images, Video, Audio, PDFs** | Preview |

Both models support **task-type optimization**:
- `RETRIEVAL_DOCUMENT` — for indexing documents
- `RETRIEVAL_QUERY` — for search queries
- `SEMANTIC_SIMILARITY` — for comparing texts
- `CLASSIFICATION` / `CLUSTERING` — for ML tasks
- `QUESTION_ANSWERING` / `FACT_VERIFICATION` — for QA pipelines
- `CODE_RETRIEVAL_QUERY` — for code search

> **Note:** Embedding spaces between models are **incompatible** — you cannot compare embeddings from different models.

Let's test the embeddings directly using the `google-genai` SDK.

In [ ]:
from google import genai
from google.genai import types
import numpy as np

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# --- Embed documents with gemini-embedding-001 ---
documents = [
    "Retrieval-Augmented Generation combines search with language models.",
    "Vector databases store high-dimensional embeddings for similarity search.",
    "The stock market closed higher today driven by tech sector gains.",
]

doc_result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=documents,
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT",
        output_dimensionality=768,
    ),
)

# --- Embed query ---
query = "How does RAG work?"

query_result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=query,
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_QUERY",
        output_dimensionality=768,
    ),
)

# --- Compute cosine similarity ---
query_vec = np.array(query_result.embeddings[0].values)
for i, emb in enumerate(doc_result.embeddings):
    doc_vec = np.array(emb.values)
    similarity = np.dot(query_vec, doc_vec) / (
        np.linalg.norm(query_vec) * np.linalg.norm(doc_vec)
    )
    print(f"Doc {i} (similarity: {similarity:.4f}): {documents[i][:60]}...")

print(f"\nEmbedding dimensions: {len(query_result.embeddings[0].values)}")

Doc 0 (similarity: 0.6864): Retrieval-Augmented Generation combines search with language...
Doc 1 (similarity: 0.5503): Vector databases store high-dimensional embeddings for simil...
Doc 2 (similarity: 0.5701): The stock market closed higher today driven by tech sector g...

Embedding dimensions: 768


---
## 4. Multimodal Embeddings with `gemini-embedding-2-preview`

`gemini-embedding-2-preview` is **Google's first multimodal embedding model** — it can embed text, images, video, audio, and PDFs into the same vector space. This enables **cross-modal search** (e.g., find images using text queries).

In [ ]:
from google import genai
from google.genai import types
import numpy as np
import urllib.request

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# --- Text embeddings with the multimodal model ---
text_result = client.models.embed_content(
    model="gemini-embedding-2-preview",
    contents="A golden retriever playing in the park",
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT",
        output_dimensionality=768,
    ),
)

print(f"Text embedding dimensions: {len(text_result.embeddings[0].values)}")
print(f"First 5 values: {text_result.embeddings[0].values[:5]}")

Text embedding dimensions: 768
First 5 values: [0.0014976152, -0.01584851, 0.004873589, 0.0044438187, 0.043795828]


In [ ]:
# --- Image embedding with gemini-embedding-2-preview ---
# Download a sample image
image_url = "https://storage.googleapis.com/generativeai-downloads/images/scones.jpg"
urllib.request.urlretrieve(image_url, "sample_image.jpg")

with open("sample_image.jpg", "rb") as f:
    image_bytes = f.read()

# Embed the image
image_result = client.models.embed_content(
    model="gemini-embedding-2-preview",
    contents=[
        types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"),
    ],
    config=types.EmbedContentConfig(output_dimensionality=768),
)

print(f"Image embedding dimensions: {len(image_result.embeddings[0].values)}")

# --- Cross-modal similarity: compare text queries to the image ---
text_queries = [
    "freshly baked scones with tea",
    "a plate of pastries and baked goods",
    "programming in Python",
]

query_results = client.models.embed_content(
    model="gemini-embedding-2-preview",
    contents=text_queries,
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_QUERY",
        output_dimensionality=768,
    ),
)

image_vec = np.array(image_result.embeddings[0].values)
print("\nCross-modal similarity (text query vs image):")
for i, emb in enumerate(query_results.embeddings):
    query_vec = np.array(emb.values)
    sim = np.dot(query_vec, image_vec) / (
        np.linalg.norm(query_vec) * np.linalg.norm(image_vec)
    )
    print(f"  '{text_queries[i]}' → similarity: {sim:.4f}")

Image embedding dimensions: 768

Cross-modal similarity (text query vs image):
  'freshly baked scones with tea' → similarity: 0.4206
  'a plate of pastries and baked goods' → similarity: 0.3140
  'programming in Python' → similarity: 0.2504


---
## 5. Agno's Gemini Embedder

Agno provides a built-in `GeminiEmbedder` that wraps Google's embedding API seamlessly. This is what we'll use in our RAG pipeline.

In [ ]:
from agno.knowledge.embedder.google import GeminiEmbedder

# Initialize the Gemini embedder with the stable model
embedder = GeminiEmbedder(
    dimensions=768,
)

# Test embedding
embedding = embedder.get_embedding("Agno makes building AI agents simple.")
print(f"Embedding type: {type(embedding)}")
print(f"Embedding dimensions: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Embedding type: <class 'list'>
Embedding dimensions: 768
First 5 values: [-0.028871268, 0.011327151, 0.03519314, -0.073764294, -0.018630492]


---
## 6. Building an Agentic RAG Pipeline

Now let's build the full RAG pipeline:

1. **Load** documents into a knowledge base
2. **Chunk** and **embed** them using `gemini-embedding-001`
3. **Store** vectors in LanceDB (serverless — no setup needed)
4. **Query** using an Agno agent that autonomously decides when to search

### 6.1 Prepare Sample Documents

In [ ]:
import os

# Create a data directory with sample text files for the knowledge base
os.makedirs("data/docs", exist_ok=True)

sample_docs = {
    "data/docs/rag_overview.txt": """Retrieval-Augmented Generation (RAG) Overview

Retrieval-Augmented Generation (RAG) is a technique that enhances large language models (LLMs)
by retrieving relevant information from external knowledge sources before generating a response.

Key Components of RAG:
1. Document Ingestion: Documents are split into chunks, converted to vector embeddings, and
   stored in a vector database.
2. Retrieval: When a user asks a question, the query is embedded and used to find the most
   similar document chunks via vector similarity search.
3. Augmented Generation: The retrieved context is combined with the user's question and sent
   to the LLM, which generates an informed, grounded response.

Benefits of RAG:
- Reduces hallucinations by grounding responses in real data
- Enables LLMs to access up-to-date or domain-specific information
- More cost-effective than fine-tuning for many use cases
- Provides transparency through source attribution

Common RAG Architectures:
- Naive RAG: Simple retrieve-then-generate pipeline
- Advanced RAG: Adds query rewriting, re-ranking, and hybrid search
- Agentic RAG: The LLM autonomously decides when and how to search the knowledge base
""",
    "data/docs/gemini_embeddings.txt": """Google Gemini Embedding Models

Google provides two embedding models in the Gemini API:

gemini-embedding-001:
- Google's stable, production-ready text embedding model
- Output dimensions: 128 to 3,072 (recommended: 768, 1536, 3072)
- Max input: 2,048 tokens
- Modalities: Text only
- Supports task-type optimization: RETRIEVAL_DOCUMENT, RETRIEVAL_QUERY,
  SEMANTIC_SIMILARITY, CLASSIFICATION, CLUSTERING, QUESTION_ANSWERING,
  FACT_VERIFICATION, CODE_RETRIEVAL_QUERY

gemini-embedding-2-preview:
- Google's first multimodal embedding model
- Output dimensions: 128 to 3,072 (recommended: 768, 1536, 3072)
- Max input: 8,192 tokens
- Modalities: Text, images (up to 6), audio (up to 80s), video (up to 128s), PDFs (up to 6 pages)
- Supports the same task types as gemini-embedding-001
- Enables cross-modal search (e.g., find images using text queries)

Important: The embedding spaces between the two models are incompatible. You cannot
directly compare embeddings generated by one model with embeddings from the other.

Best Practices:
- Use RETRIEVAL_DOCUMENT task type when indexing documents
- Use RETRIEVAL_QUERY task type when embedding user queries
- For dimensions below 3,072, normalize embeddings after generation
- Chunk documents to 500-1000 tokens for optimal retrieval
- Use overlapping chunks (100-200 tokens) to preserve context at boundaries
""",
    "data/docs/agno_framework.txt": """Agno: A Lightweight AI Agent Framework

Agno is an open-source framework for building multi-modal AI agents. It provides a clean,
Pythonic API for creating agents that can reason, use tools, and access knowledge bases.

Core Concepts:
1. Agent: The central abstraction that wraps an LLM with tools, knowledge, and instructions.
2. Model: Supports multiple providers — OpenAI, Google Gemini, Anthropic, and more.
3. Tools: Extend agent capabilities (web search, file operations, API calls, etc.).
4. Knowledge Base: Connect agents to vector databases for RAG.
5. Team: Coordinate multiple specialized agents for complex tasks.

Key Features:
- Model-agnostic: Switch between LLM providers with a single line change
- Built-in RAG: First-class support for knowledge bases and vector stores
- Agentic search: Agents autonomously decide when to query the knowledge base
- Structured output: Return Pydantic models for type-safe responses
- Multi-agent teams: Orchestrate specialized agents for divide-and-conquer workflows
- Streaming: Real-time token streaming for responsive UIs
""",
}

for path, content in sample_docs.items():
    with open(path, "w") as f:
        f.write(content)

print(f"Created {len(sample_docs)} sample documents in data/docs/")

Created 3 sample documents in data/docs/


### 6.2 Create the Knowledge Base with Gemini Embeddings + LanceDB

In [ ]:
import asyncio
from agno.knowledge.embedder.google import GeminiEmbedder
from agno.vectordb.lancedb import LanceDb, SearchType
from agno.knowledge.knowledge import Knowledge

# 1. Setup the Vector DB
vector_db = LanceDb(
    uri="tmp/lancedb",
    table_name="agno_gemini_docs",
    search_type=SearchType.vector,
    embedder=GeminiEmbedder(),
)

# 2. Setup the Knowledge Base
knowledge_base = Knowledge(
    vector_db=vector_db,
    max_results=2,
)

# 3. In Colab/Jupyter, we can use top-level await
# Instead of asyncio.run(main()), we call the async methods directly
print("Loading documents into the knowledge base...")

# We point it to the directory containing our created text files
await knowledge_base.ainsert(path="data/docs")

print("Knowledge base loaded successfully!")

### 6.3 Create the RAG Agent

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini

rag_agent = Agent(
    model=Gemini(id="gemini-2.5-flash"),
    knowledge=knowledge_base,
    search_knowledge=True,  # Agent autonomously decides when to search
    description="You are an AI expert that answers questions using the knowledge base.",
    instructions=[
        "Always search the knowledge base for relevant information before answering.",
        "Cite which document your answer came from.",
        "If the knowledge base doesn't contain the answer, say so clearly.",
    ],
    markdown=True,
)

In [ ]:
# Query 1: About RAG
rag_agent.print_response(
    "What are the three main components of a RAG pipeline?",
    stream=True,
)

In [ ]:
# Query 2: About Gemini Embedding models
rag_agent.print_response(
    "What is the difference between gemini-embedding-001 and gemini-embedding-2-preview?",
    stream=True,
)

In [ ]:
# Query 3: About Agno
rag_agent.print_response(
    "What are the core concepts of the Agno framework?",
    stream=True,
)

---
## 8. Agent with Tools + Knowledge Base

Agno agents can combine RAG with external tools. Here we create an agent that can both search its knowledge base **and** search the web.

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini
from agno.tools.duckduckgo import DuckDuckGoTools

agent = Agent(tools=[DuckDuckGoTools()])
research_agent = Agent(
    model=Gemini(id="gemini-2.5-flash"),
    knowledge=knowledge_base,
    search_knowledge=True,
    tools=[DuckDuckGoTools()],
    description="You are an AI research assistant with access to a knowledge base and web search.",
    instructions=[
        "First search the knowledge base for relevant information.",
        "Use web search to find additional or more recent information.",
        "Combine both sources to give comprehensive answers.",
        "Always cite your sources.",
    ],
    markdown=True,
)

research_agent.print_response(
    "What is Agentic RAG and how does it compare to traditional RAG? Include recent developments.",
    stream=True,
)

### Multimodel RAG With Gemini Embedding 2

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini
from agno.knowledge.embedder.google import GeminiEmbedder
from agno.knowledge.knowledge import Knowledge
from agno.vectordb.lancedb import LanceDb, SearchType

# 1. Setup Knowledge Base
knowledge_base = Knowledge(
    vector_db=LanceDb(
        uri="tmp/lancedb",
        table_name="gemini_docs",
        search_type=SearchType.vector,
        embedder=GeminiEmbedder(id="gemini-embedding-2-preview")
    ),
)

# 2. Insert knowledge from a directory
knowledge_base.insert(
    name="Gemini Docs",
    url="https://ai.google.dev/gemini-api/docs/embeddings",
)

# 3. Create the Agent
agent = Agent(
    model=Gemini(id="gemini-2.5-flash"),
    knowledge=knowledge_base,
    search_knowledge=True,
    markdown=True,
)

agent.print_response("What is unique about Gemini Embedding 2?", stream=True)

---
## 10. Multi-Agent Team

Agno's `Team` class lets you orchestrate multiple specialized agents. Here we create a research team with:
- A **Knowledge Agent** that searches the knowledge base
- A **Web Agent** that searches the internet
- A **lead model** that coordinates them

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.team import Team

# Agent 1: Knowledge base specialist
kb_agent = Agent(
    name="Knowledge Base Expert",
    role="Search the internal knowledge base for information",
    model=Gemini(id="gemini-2.5-flash"),
    knowledge=knowledge_base,
    search_knowledge=True,
    instructions=[
        "Always search the knowledge base.",
        "Provide detailed answers based on the documents.",
    ],
    markdown=True,
)

# Agent 2: Web search specialist
web_agent = Agent(
    name="Web Researcher",
    role="Search the web for the latest information",
    model=Gemini(id="gemini-2.5-flash"),
    tools=[DuckDuckGoTools()],
    instructions=[
        "Search the web for recent and relevant information.",
        "Always include sources in your response.",
    ],
    markdown=True,
)

# Coordinate with a Team using 'members'
research_team = Team(
    members=[kb_agent, web_agent],
    model=Gemini(id="gemini-2.5-flash"),
    instructions=[
        "Delegate knowledge base questions to the Knowledge Base Expert.",
        "Delegate web search questions to the Web Researcher.",
        "Synthesize information from both agents into a comprehensive answer.",
    ],
    markdown=True,
)

research_team.print_response(
    "Give me a comprehensive overview of Gemini embedding models for RAG — "
    "what does our knowledge base say, and what are the latest developments on the web?",
    stream=True,
)

---
## 11. Comparing Embedding Models

Let's compare `gemini-embedding-001` (stable) with `gemini-embedding-2-preview` (multimodal) on a text retrieval task.

In [ ]:
from google import genai
from google.genai import types
import numpy as np

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])


def compute_similarities(model_name, query, documents, dim=768):
    """Embed query and documents, then compute cosine similarities."""
    query_result = client.models.embed_content(
        model=model_name,
        contents=query,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",
            output_dimensionality=dim,
        ),
    )
    doc_result = client.models.embed_content(
        model=model_name,
        contents=documents,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_DOCUMENT",
            output_dimensionality=dim,
        ),
    )

    query_vec = np.array(query_result.embeddings[0].values)
    similarities = []
    for emb in doc_result.embeddings:
        doc_vec = np.array(emb.values)
        sim = np.dot(query_vec, doc_vec) / (
            np.linalg.norm(query_vec) * np.linalg.norm(doc_vec)
        )
        similarities.append(sim)

    return similarities, len(query_result.embeddings[0].values)


# Test data
query = "How do embedding models help with search?"
documents = [
    "Embedding models convert text to vectors for semantic similarity search.",  # Highly relevant
    "Neural networks can be trained on large datasets for various tasks.",       # Somewhat relevant
    "The weather forecast predicts rain for the upcoming weekend.",              # Not relevant
    "Vector databases like LanceDB store embeddings for fast retrieval.",        # Highly relevant
]

# Compare models at the same dimensionality
models = [
    "gemini-embedding-001",
    "gemini-embedding-2-preview",
]

for model_name in models:
    try:
        sims, dims = compute_similarities(model_name, query, documents, dim=768)
        print(f"\nModel: {model_name} (dims={dims})")
        print("-" * 60)
        for i, (sim, doc) in enumerate(zip(sims, documents)):
            print(f"  Doc {i} ({sim:.4f}): {doc[:55]}...")
        ranked = sorted(range(len(sims)), key=lambda i: sims[i], reverse=True)
        print(f"  Ranking: {['Doc ' + str(r) for r in ranked]}")
    except Exception as e:
        print(f"\nModel: {model_name} — Error: {e}")


Model: gemini-embedding-001 (dims=768)
------------------------------------------------------------
  Doc 0 (0.7916): Embedding models convert text to vectors for semantic s...
  Doc 1 (0.6058): Neural networks can be trained on large datasets for va...
  Doc 2 (0.5212): The weather forecast predicts rain for the upcoming wee...
  Doc 3 (0.7282): Vector databases like LanceDB store embeddings for fast...
  Ranking: ['Doc 0', 'Doc 3', 'Doc 1', 'Doc 2']

Model: gemini-embedding-2-preview (dims=768)
------------------------------------------------------------
  Doc 0 (0.8759): Embedding models convert text to vectors for semantic s...
  Doc 1 (0.6450): Neural networks can be trained on large datasets for va...
  Doc 2 (0.4511): The weather forecast predicts rain for the upcoming wee...
  Doc 3 (0.7515): Vector databases like LanceDB store embeddings for fast...
  Ranking: ['Doc 0', 'Doc 3', 'Doc 1', 'Doc 2']


---
## 12. Cleanup

In [ ]:
import shutil

# Clean up local data
for path in ["tmp/lancedb", "data/docs", "sample_image.jpg"]:
    if os.path.isdir(path):
        shutil.rmtree(path, ignore_errors=True)
    elif os.path.isfile(path):
        os.remove(path)

print("Cleanup complete!")

Cleanup complete!


###

---
## Summary

In this notebook, we demonstrated:

| Section | What We Built |
|---|---|
| **Quick Start** | A simple Agno agent powered by Gemini 2.5 Flash |
| **Gemini Embeddings** | Direct usage of `gemini-embedding-001` with task-type optimization |
| **Multimodal Embeddings** | Image embedding and cross-modal search with `gemini-embedding-2-preview` |
| **Agno Embedder** | Agno's built-in `GeminiEmbedder` wrapper |
| **Agentic RAG** | Full RAG pipeline with LanceDB + Gemini embeddings |
| **PDF RAG** | Loading and querying research papers from URLs |
| **Tools + RAG** | Agent combining knowledge base search with web search |
| **Structured Output** | Extracting typed data from the knowledge base |
| **Multi-Agent Team** | Coordinating specialized agents for complex research |
| **Model Comparison** | Comparing `gemini-embedding-001` vs `gemini-embedding-2-preview` |

### Resources

- [Agno Documentation](https://docs.agno.com)
- [Agno GitHub](https://github.com/agno-agi/agno)
- [Google Gemini Embeddings](https://ai.google.dev/gemini-api/docs/embeddings)
- [Google AI Studio](https://aistudio.google.com/) — Get your API key
- [LanceDB Documentation](https://lancedb.github.io/lancedb/)

---

*Built with [Agno](https://github.com/agno-agi/agno) and [Google Gemini](https://ai.google.dev/)*